[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/29_adam.ipynb)

# 🟠 中等：Adam 优化器

从零实现 **Adam** 优化器。

### 函数签名
```python
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8): ...
    def step(self): ...
    def zero_grad(self): ...
```

### 算法 (per parameter)
```
m = β1 * m + (1-β1) * grad
v = β2 * v + (1-β2) * grad²
m̂ = m / (1 - β1ᵗ)    # bias correction
v̂ = v / (1 - β2ᵗ)
p -= lr * m̂ / (√v̂ + ε)
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ 在此实现你的代码

class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        pass  # 存储 params，初始化 m 和 v 为零

    def step(self):
        pass  # 使用 Adam 规则更新 params

    def zero_grad(self):
        pass  # 清零所有梯度

#### <strong>批量梯度下降(BGD, Batch Gradient Descent)</strong>的核心公式非常简单，就是对整个训练集计算损失函数的梯度，然后更新参数。具体数学表达如下：

**1. 参数更新公式（通用形式）**

$\theta = \theta - \eta \cdot abla J(\theta)$

- **$\theta$**：模型的参数向量（权重和偏置）。
- **$\eta$（学习率）**：控制更新步长的超参数。
- **$
abla J(\theta)$**：损失函数 $J$ 关于参数 $\theta$ 的梯度。

**2. 批量梯度下降法的迭代更新公式**
假设线性回归的假设函数为  $h_\theta(x^{(i)}) = \theta^T x^{(i)}$，损失函数为均方误差（MSE）：
$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} \left( h_\theta(x^{(i)}) - y^{(i)} \right)^2$

那么，**BGD 的迭代更新公式**为：
$\theta_j = \theta_j - \eta \cdot \frac{1}{m} \sum_{i=1}^{m} \left( h_\theta(x^{(i)}) - y^{(i)} \right) x_j^{(i)}$

**关键点解读：**

- **$m$** 是训练集样本总数。
- **$j$** 表示第 $j$ 个特征维度。
- **求和符号 $\sum_{i=1}^{m}$** 意味着**每一次参数更新**，都必须遍历**全部 $m$ 个样本**来计算梯度。

---
#### <strong>随机梯度下降(SGD, Stochastic Gradient Descent)</strong>的核心思想与 BGD 恰好相反：**每次参数更新时，不再计算整个数据集的梯度，而是仅随机抽取一个样本 $(x^{(i)}, y^{(i)})$ 来计算梯度**。

**1. 参数更新公式（通用形式）**

$\theta = \theta - \eta \cdot abla J_i(\theta)$

- **$\theta$**：模型参数。
- **$\eta$**（学习率）：控制步长。
- **$abla J_i(\theta)$**：**仅基于第$i$个样本**计算出的损失函数梯度。

**2. 随机梯度下降法的迭代更新公式**

假设线性回归的损失函数为均方误差（针对**单个样本** $i$）
$J_i(\theta) = \frac{1}{2} \left( h_\theta(x^{(i)}) - y^{(i)} \right)^2$
那么，SGD 对第 $j$ 个参数的**迭代更新公式**为：
$\theta_j = \theta_j - \eta \cdot \left( h_\theta(x^{(i)}) - y^{(i)} \right) \cdot x_j^{(i)}$

**关键点解读：**

- **没有求和符号 $\sum $**：公式中只出现第 $i$ 个样本的特征 $x_j^{(i)}$ 和标签 $y^{(i)}$。
- **更新频率极高**：每处理 **1 个**样本，参数就更新一次。遍历完整个数据集（1个 Epoch）时，参数已经更新了 m 次（ $m$ 为样本总数）。

**3. 与 BGD 的本质区别（核心对比）**

| 对比维度 | **SGD（随机梯度下降）** | **BGD（批量梯度下降）** |
| :--- | :--- | :--- |
| **每次更新所用样本** | 1 个样本 | 全部 $m $个样本 |
| **更新公式中的运算** | 无求和，单点计算 | 包含 $\sum_{i=1}^{m}$ 全量求和 |
| **计算速度** | 极快（单次迭代） | 极慢（大数据集下） |
| **收敛路径** | 震荡剧烈，呈“锯齿状” | 平滑稳定，笔直走向最优点 |
| **收敛效果** | 无法收敛到精确最小值，会在最优点附近波动 | 能稳定收敛到全局最优点（凸函数） |
| **跳出局部最优** | **更容易**（震荡特性有助于逃离局部极小点和鞍点） | 容易陷入局部最优 |

**4. 一个重要补充：实际工程中的“SGD”**

在现代深度学习框架（如 PyTorch、TensorFlow）中，当你调用 `torch.optim.SGD` 时，**它通常并不是严格的“1个样本更新一次”**。

实际使用时，你传入的往往是 **一个 mini-batch（小批量）** 的数据（例如 32、64 或 128 个样本）。框架会计算这一个小批量的平均梯度来更新参数。这种做法的数学公式为：

$\theta = \theta - \eta \cdot \frac{1}{B} \sum_{i=1}^{B} abla J_i(\theta)$

（其中 $B$ 是 batch size，即小批量大小）

这本质上是 **小批量梯度下降（MBGD）**，但由于工程习惯，大家仍统称它为 SGD。如果想严格使用“1个样本”的纯 SGD，只需将 `batch_size` 设为 1 即可。

---

#### **动量法（SGD with Momentum）** 
动量法的核心思想是模拟物理中的动量概念，在梯度和参数更新中间维护一个动量状态$v_t$(保持一定的历史梯度信息)。当梯度方向一致时，累积速度会越来越快，从而加速收敛；当遇到局部极小点或鞍点时，积累的动量可以根据动量跳过去，避免陷入困境，逃离局部最优。

**1. 参数更新公式（通用形式）**

$v_t = \gamma v_{t-1} + \eta 
abla J(\theta)$

$\theta = \theta - v_t$

-   **$\theta$**：模型的参数向量。
-   **$\eta$（学习率）**：控制步长的超参数。
-   **$
abla J(\theta)$**：当前批次计算出的损失函数梯度。
-   **$v$（速度/动量）**：累积的梯度动量，初始化为 0。
-   **$\gamma$（动量衰减系数）**：通常取 0.9，用于控制历史梯度对当前速度的影响程度。

**2. 动量法的迭代更新公式**

以线性回归的均方误差损失为例，参数 $\theta_j$ 的更新过程包含两个步骤：

**第一步：计算当前梯度并更新动量**
$v_j = \gamma v_j + \eta \cdot \frac{1}{m} \sum_{i=1}^{m} \left( h_\theta(x^{(i)}) - y^{(i)} \right) x_j^{(i)}$

**第二步：更新参数**
$\theta_j = \theta_j - v_j$

**关键点解读：**

-   **累积历史梯度**：$v_j$ 不仅包含当前梯度，还通过 $\gamma$ 保留了历史梯度的方向和信息。
-   **加速与抑制**：在梯度方向一致的平坦区域，$v_j$ 会不断增大，实现“加速”；在梯度方向频繁改变的区域（如狭窄山谷），$v_j$ 会相互抵消，抑制“震荡”。
-   **冲出局部最优**：当梯度为 0（如鞍点）时，$v_j$ 不会立即归零，而是带着已有的速度继续前进，使其更容易逃离局部极小点和鞍点。

---

#### **自适应梯度（AdaGrad）**

AdaGrad 的核心思想是为每个参数单独维护一个学习率。其原理是：在训练过程中，对于频繁更新的参数，降低其学习率，使其步长变小,对于稀疏更新的参数，增大其学习率，使其步长变大,这使其特别适合处理稀疏数据（如文本分类、推荐系统）。

**1. 参数更新公式（通用形式）**

$G_t = G_{t-1} + (
abla J(\theta))^2$

$\theta = \theta - \frac{\eta}{\sqrt{G_t + \epsilon}} \odot 
abla J(\theta)$

-   **$\theta$**：模型的参数向量。
-   **$\eta$（学习率）**：全局基础学习率。
-   **$
abla J(\theta)$**：当前梯度。
-   **$G_t$**：梯度平方的累积和，是一个与参数同维度的向量。
-   **$\epsilon$**：一个极小常数（如 $10^{-8}$），用于防止分母为零。
-   **$\odot$**：逐元素相乘（Hadamard 积）。

**2. AdaGrad 的迭代更新公式**

对于第 $j$ 个参数 $\theta_j$，其更新过程如下：

**第一步：累积历史梯度平方和**
$G_{j} = G_{j} + \left( \frac{1}{m} \sum_{i=1}^{m} \left( h_\theta(x^{(i)}) - y^{(i)} \right) x_j^{(i)} \right)^2$

**第二步：自适应调整学习率并更新参数**
$\theta_j = \theta_j - \frac{\eta}{\sqrt{G_j + \epsilon}} \cdot \frac{1}{m} \sum_{i=1}^{m} \left( h_\theta(x^{(i)}) - y^{(i)} \right) x_j^{(i)}$

**关键点解读：**

-   **自适应学习率**：每个参数 $\theta_j$ 都有自己独立的学习率 $\eta / \sqrt{G_j + \epsilon}$。
-   **惩罚高频**：如果一个参数的历史梯度 $G_j$ 很大，其学习率会被大幅缩小；反之，如果历史梯度很小，其学习率会相对较大。
-   **无需手动调整**：大大减少了手动调整学习率的负担。
-   **致命缺点**：$G_t$ 是单调递增的，导致学习率 $\eta / \sqrt{G_t + \epsilon}$ 会持续衰减，最终趋近于 0，导致模型停止学习（“学习率归零”问题）。

---

#### **均方根传播（RMSProp）**

RMSProp 针对 AdaGrad 学习率单调递减的致命缺陷进行了改进。它引入了一个**衰减系数 $\beta$**（或称平滑常）,使历史梯度的影响随时间指数级衰减,只关注最近一段时间的梯度大小,这保证了学习率始终保持“活性”，不会提前归零，从而在非凸优化（如深度学习）中表现更稳定。

**1. 参数更新公式（通用形式）**

$G_t = \beta G_{t-1} + (1 - \beta) (
abla J(\theta))^2$

$\theta = \theta - \frac{\eta}{\sqrt{G_t + \epsilon}} \odot 
abla J(\theta)$

-   **$\theta$**：模型的参数向量。
-   **$\eta$（学习率）**：全局基础学习率。
-   **$
abla J(\theta)$**：当前梯度。
-   **$G_t$**：梯度平方的**指数加权移动平均**（EWMA）。
-   **$\beta$（衰减系数）**：通常取 0.9 或 0.99，控制历史信息的衰减速度。
-   **$\epsilon$**：极小常数（如 $10^{-8}$），用于数值稳定。

**2. RMSProp 的迭代更新公式**

对于第 $j$ 个参数 $\theta_j$，其更新过程如下：

**第一步：计算梯度平方的指数加权移动平均**
$G_j = \beta G_j + (1 - \beta) \left( \frac{1}{m} \sum_{i=1}^{m} \left( h_\theta(x^{(i)}) - y^{(i)} \right) x_j^{(i)} \right)^2$

**第二步：自适应调整学习率并更新参数**
$\theta_j = \theta_j - \frac{\eta}{\sqrt{G_j + \epsilon}} \cdot \frac{1}{m} \sum_{i=1}^{m} \left( h_\theta(x^{(i)}) - y^{(i)} \right) x_j^{(i)}$

**关键点解读：**

-   **指数衰减**：$(1 - \beta)$ 给近期梯度更高权重，而 $\beta$ 的幂次方使很久以前的梯度影响近乎为零，解决了 AdaGrad 的历史累积问题。
-   **学习率活性**：由于 $G_j$ 不再单调递增，学习率 $\eta / \sqrt{G_j + \epsilon}$ 能根据梯度近期变化动态调整，不会出现“冻结”现象。
-   **适应性学习率**：依然保留了 AdaGrad 为每个参数单独设置学习率的优点。
-   **非凸优化首选**：非常适合深度学习中的非凸损失函数，是目前最常用的基础优化器之一。

---

#### **自适应矩估计（Adam）**

Adam（Adaptive Moment Estimation）是目前深度学习领域最主流、最常用的优化算法。它巧妙地融合了动量法（Momentum）和 RMSProp 的核心思想：既像 Momentum 一样利用**一阶矩**（梯度的指数加权平均）来积累历史动量和加速收敛,又像 RMSProp 一样利用**二阶矩**（梯度平方的指数加权平均）来为每个参数自适应调整学习率。

**1. 参数更新公式（通用形式）**

**第一步：计算动量（一阶矩）和自适应学习率（二阶矩）**

$m_t = \beta_1 m_{t-1} + (1 - \beta_1) 
abla J(\theta)$

$v_t = \beta_2 v_{t-1} + (1 - \beta_2) (
abla J(\theta))^2$

**第二步：偏差修正（防止初始阶段偏向 0）**
$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$

$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$

**第三步：更新参数**
$\theta = \theta - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$

-   **$\theta$**：模型的参数向量。
-   **$\eta$（学习率）**：全局基础学习率。
-   **$
abla J(\theta)$**：当前梯度。
-   **$m_t$**：梯度的一阶矩（均值，即动量）估计。
-   **$v_t$**：梯度的二阶矩（方差，即未中心化的方差）估计。
-   **$\beta_1$**：一阶矩衰减系数，通常取 0.9。
-   **$\beta_2$**：二阶矩衰减系数，通常取 0.999。
-   **$\epsilon$**：极小常数（如 $10^{-8}$），用于防止分母为零。
-   **$t$**：当前迭代步数，用于偏差修正。

**2. Adam 的迭代更新公式**

对于第 $j$ 个参数 $\theta_j$，其完整更新过程如下：

**第一步：计算当前梯度**
$g_j = \frac{1}{m} \sum_{i=1}^{m} \left( h_\theta(x^{(i)}) - y^{(i)} \right) x_j^{(i)}$

**第二步：更新一阶矩和二阶矩**

$m_j = \beta_1 m_j + (1 - \beta_1) g_j$

$v_j = \beta_2 v_j + (1 - \beta_2) g_j^2$

**第三步：偏差修正 (重要，防止训练初期一阶矩和二阶矩接近于 0)**

$\hat{m}_j = \frac{m_j}{1 - \beta_1^t}$

$\hat{v}_j = \frac{v_j}{1 - \beta_2^t}$

**第四步：更新参数**
$\theta_j = \theta_j - \eta \cdot \frac{\hat{m}_j}{\sqrt{\hat{v}_j} + \epsilon}$

**关键点解读：**

-   **双重视角**：$m_j$ 提供动量方向，加速收敛并克服震荡；$v_j$ 提供自适应步长，让每个参数有独立的更新尺度。
-   **偏差修正**：由于 $m$ 和 $v$ 初始化为 0，在训练初期会偏向 0，$1 - \beta^t$ 项能有效校正这一偏差，使起步更新更稳定。
-   **鲁棒性强**：对超参数（尤其是学习率 $\eta$）的选择不太敏感，默认参数（$\eta=0.001, \beta_1=0.9, \beta_2=0.999$）在绝大多数问题上表现良好。
-   **计算高效**：所需内存适中，计算量小，非常适合大规模数据集和高维参数空间（如深度神经网络）。

**3. 与 SGD / Momentum / RMSProp 的本质区别（核心对比）**

| 对比维度 | **Adam** | **SGD** | **Momentum** | **RMSProp** |
| :--- | :--- | :--- | :--- | :--- |
| **核心机制** | **动量 + 自适应学习率** | 纯梯度下降 | 纯动量累积 | 纯自适应学习率 |
| **一阶矩 ($m$)** | **使用**，并做偏差修正 | 不使用 | **使用**，无修正 | 不使用 |
| **二阶矩 ($v$)** | **使用**，并做偏差修正 | 不使用 | 不使用 | **使用**，无修正 |
| **学习率** | **每个参数自适应**，且动态变化 | 全局固定 | 全局固定 | **每个参数自适应**，动态变化 |
| **收敛速度** | **极快**，综合了两者优势 | 慢 | 较快 | 快 |
| **稳定性** | **非常稳定**，超参数宽容度高 | 震荡大，不稳定 | 较稳定 | 稳定，但超参数敏感度高于 Adam |
| **泛化能力** | 通常在 CV/NLP 表现优异 | 某些任务上泛化可能更好 | 中等 | 中等 |
| **内存占用** | **较高**（需存储 $m$ 和 $v$） | 低 | 中（仅存储 $v$） | 中（仅存储 $G$） |


In [ ]:
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        """
        Adam 优化器
        
        Args:
            params: 可迭代的模型参数（要求每个参数有 .data 和 .grad 属性）
            lr: 学习率
            betas: (beta1, beta2)，一阶矩和二阶矩的衰减率
            eps: 数值稳定项，防止分母为零
        """
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.t = 0  # 时间步
        
        # 为每个参数初始化一阶矩和二阶矩估计
        self.m = [None] * len(self.params)  # 一阶矩（均值） mean
        self.v = [None] * len(self.params)  # 二阶矩（方差） variance
        
        for i, p in enumerate(self.params):
            self.m[i] = p.data.new_zeros(p.data.shape)  # 创建与参数形状相同的零张量
            self.v[i] = p.data.new_zeros(p.data.shape)
    
    def zero_grad(self):
        """将所有参数的梯度清零"""
        for p in self.params:
            if p.grad is not None:
                p.grad.data.zero_()
    
    def step(self):
        """
        执行一步参数更新
        使用 Adam 算法更新所有参数
        """
        self.t += 1  # 增加时间步
        
        # 计算偏差校正系数
        beta1_pow = self.beta1 ** self.t
        beta2_pow = self.beta2 ** self.t
        
        for i, p in enumerate(self.params):
            # 确保参数有梯度
            if p.grad is None:
                continue
            
            grad = p.grad.data
            
            # 更新一阶矩估计（动量）
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * grad
            
            # 更新二阶矩估计（RMS）
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (grad * grad)
            
            # 偏差校正
            m_hat = self.m[i] / (1 - beta1_pow)
            v_hat = self.v[i] / (1 - beta2_pow)
            
            # 更新参数
            p.data -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

In [ ]:
# 🧪 调试
torch.manual_seed(0)
w = torch.randn(4, 3, requires_grad=True)
opt = MyAdam([w], lr=0.01)
for i in range(5):
    loss = (w ** 2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f'步骤 {i}: 损失={loss.item():.4f}')

In [ ]:
# ✅ 提交
from torch_judge import check
check('adam')